# Pareja A · Idea 7a — Nuevos codificadores (versión Kaggle)

Lo mismo que `A7_codificadores.ipynb` (Colab), pero en Kaggle: vectores de CLIP L/14 y SigLIP base y
los experimentos `A7_l14` y `A7_siglip`. Sirve para usar la GPU de Kaggle (30 h por semana) cuando se
acaba la de Colab.

**Diferencia con Colab:** Kaggle no puede montar Google Drive. Este notebook usa
[rclone](https://rclone.org/drive/) para:

1. **Bajar** el dataset (`poster2movie_datos`, ~3.3 GB de `.parquet`) al disco de Kaggle.
2. **Bajar** los pedazos de vectores que ya estén en `poster2movie_clasificador/vectores/partes/` (si
   alguien empezó en Colab o en otra corrida, se continúa desde ahí). Si los vectores completos ya
   están en el Drive, no se vuelven a calcular.
3. **Subir solo** al Drive del equipo (`poster2movie_clasificador/`) los vectores, modelos y
   predicciones: cada 10 minutos mientras calcula y al terminar cada paso. Las otras parejas los
   ven igual que si se hubieran calculado en Colab.

Las métricas (`resultados/A7_*`) se suben a la rama `A-codificadores` con la última celda, como en
Colab (y también se copian al Drive como respaldo).

## Antes de la primera vez

**En Kaggle** (panel derecho, *Settings*): *Accelerator* → **GPU T4 x2** (o P100) e *Internet* →
**On** (pide verificar la cuenta con un teléfono).

**Llave del Drive (una vez por persona):** en tu computadora instala rclone
(<https://rclone.org/downloads/>) y corre:

```bash
rclone authorize "drive"
```

Se abre el navegador: entra con la cuenta de Google que tiene `poster2movie_datos` y
`poster2movie_clasificador` como acceso directo en *Mi unidad*. Al final la terminal imprime un texto
`{"access_token": ..., "refresh_token": ..., ...}`: cópialo completo (una línea, entre las llaves).

**En Kaggle**, *Add-ons → Secrets*:

| Secreto | Valor | ¿Obligatorio? |
| --- | --- | --- |
| `RCLONE_DRIVE_TOKEN` | el texto `{...}` de `rclone authorize` | sí |
| `GITHUB_TOKEN` | token *classic* con permiso `repo` (el mismo de Colab) | para subir `resultados/` sin que lo pida |
| `GIT_NOMBRE`, `GIT_CORREO` | tu nombre y correo de GitHub | igual |

Activa los secretos en **este** notebook (casilla junto a cada uno).

⚠️ El `RCLONE_DRIVE_TOKEN` da acceso a **todo tu Drive**: guárdalo solo en *Secrets* de Kaggle, nunca
en una celda, en el repo ni en el chat. Si se filtra, quítale el acceso a rclone en
<https://myaccount.google.com/permissions>.

**Correr sin tener la pestaña abierta:** *Save Version → Save & Run All (Commit)*. Corre en segundo
plano hasta 12 h; con los secretos `GITHUB_TOKEN`, `GIT_NOMBRE` y `GIT_CORREO` sube los resultados
solo. Si se corta, vuelve a correrlo: continúa con los pedazos que ya están en el Drive.

## Preparar el entorno

Cambia las carpetas si en tu *Mi unidad* los accesos directos tienen otro nombre.

In [ ]:
import json, os, subprocess, threading, time
from pathlib import Path

# --- Drive del equipo (rutas dentro de *Mi unidad*) ---
REMOTO = "gdrive"                          # nombre interno de rclone, no hay que cambiarlo
DRIVE_DATOS = "poster2movie_datos"          # dataset compartido (solo se LEE)
DRIVE_SALIDAS = "poster2movie_clasificador"  # carpeta del equipo (aquí se SUBEN vectores, modelos, predicciones)

# --- Repo ---
REPO = "11909/Poster2Movie-Clasificador"
RAMA = "A-codificadores"
DESTINO = "/kaggle/working/clasificador"   # en /kaggle/working: queda en la salida del notebook

# --- Carpetas locales de Kaggle ---
os.environ["P2M_DATOS"] = "/tmp/p2m_datos"                  # copia local del dataset (no va a la salida)
os.environ["P2M_SALIDAS"] = "/kaggle/working/p2m_salidas"   # se sincroniza con DRIVE_SALIDAS
for carpeta in (os.environ["P2M_DATOS"], os.environ["P2M_SALIDAS"]):
    Path(carpeta).mkdir(parents=True, exist_ok=True)

CODIFICADORES = ["clip_l14", "siglip_base"]
EXPERIMENTOS = ["A7_l14", "A7_siglip"]
EN_SEGUNDO_PLANO = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch"   # "Save & Run All"


def secreto(nombre):
    # Un secreto de Kaggle (Add-ons -> Secrets), o None si no existe o no está activado.
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(nombre)
    except Exception:
        return None


# --- rclone: instalar y configurar con la llave del Drive ---
if not Path("/usr/local/bin/rclone").exists():
    subprocess.run("curl -sSL -o /tmp/rclone.zip https://downloads.rclone.org/rclone-current-linux-amd64.zip"
                   " && unzip -q -o /tmp/rclone.zip -d /tmp/rclone"
                   " && cp /tmp/rclone/rclone-*-linux-amd64/rclone /usr/local/bin/", shell=True, check=True)

token = secreto("RCLONE_DRIVE_TOKEN")
if not token:
    raise RuntimeError("Falta el secreto RCLONE_DRIVE_TOKEN (ver 'Antes de la primera vez'): "
                       "Add-ons -> Secrets, y activarlo para este notebook.")
json.loads(token)  # si falla aquí, el secreto no es el texto {...} completo de `rclone authorize "drive"`
conf = Path.home() / ".config/rclone/rclone.conf"
conf.parent.mkdir(parents=True, exist_ok=True)
conf.write_text(f"[{REMOTO}]\ntype = drive\nscope = drive\ntoken = {token.strip()}\n")
conf.chmod(0o600)
del token


def rclone(*args, check=True):
    return subprocess.run(["rclone", *args], check=check, capture_output=True, text=True)


for carpeta in (DRIVE_DATOS, DRIVE_SALIDAS):
    r = rclone("lsf", f"{REMOTO}:{carpeta}", "--max-depth", "1", check=False)
    if r.returncode != 0:
        raise RuntimeError(f"No encuentro '{carpeta}' en tu Mi unidad. ¿Agregaste el acceso directo? "
                           f"Detalle de rclone:\n{r.stderr}")
print("Drive conectado:", DRIVE_DATOS, "y", DRIVE_SALIDAS)

In [ ]:
# --- Repo y dependencias ---
if not os.path.isdir(DESTINO):
    !git clone -q --recurse-submodules https://github.com/{REPO}.git {DESTINO}
%cd {DESTINO}
!git fetch -q origin
!git checkout -q {RAMA} 2>/dev/null || git checkout -q -b {RAMA} origin/main
!git pull -q origin {RAMA} 2>/dev/null; git submodule update -q --init
!pip install -q -r requirements.txt sentencepiece
!nvidia-smi -L

In [ ]:
# ¿ya pasa la prueba de la idea? (mientras esté pendiente sale como "xfailed")
!python -m pytest -q tests/test_pendientes.py -k idea7a -rxX

## Bajar del Drive y sincronizar

Baja el dataset y lo que ya exista de estos vectores. Después, un hilo sube `P2M_SALIDAS` al Drive
cada 10 minutos (`rclone copy`: solo agrega o actualiza, **nunca borra** nada del Drive).

In [ ]:
SALIDAS = os.environ["P2M_SALIDAS"]
DRIVE = f"{REMOTO}:{DRIVE_SALIDAS}"

print("Bajando el dataset (~3.3 GB)...")
rclone("copy", f"{REMOTO}:{DRIVE_DATOS}", os.environ["P2M_DATOS"],
       "--include", "/*.parquet", "--transfers", "8")
print(sorted(p.name for p in Path(os.environ["P2M_DATOS"]).glob("*.parquet")))

YA_CALCULADOS = []
for codificador in CODIFICADORES:
    nombre = f"{codificador}__centro"
    if rclone("lsf", f"{DRIVE}/vectores/{nombre}.npz", check=False).stdout.strip():
        rclone("copy", f"{DRIVE}/vectores", f"{SALIDAS}/vectores", "--include", f"/{nombre}.*")
        YA_CALCULADOS.append(codificador)
        print(f"{nombre}: ya está completo en el Drive, no se recalcula.")
    else:
        rclone("copy", f"{DRIVE}/vectores/partes/{nombre}", f"{SALIDAS}/vectores/partes/{nombre}", check=False)
        hechos = len(list(Path(f"{SALIDAS}/vectores/partes/{nombre}").glob("*.npz")))
        print(f"{nombre}: {hechos} pedazos ya calculados en el Drive.")

_candado = threading.Lock()
_detener = threading.Event()


def subir_al_drive():
    # vectores/, modelos/ y predicciones/ locales -> carpeta del equipo en el Drive.
    with _candado:
        r = rclone("copy", SALIDAS, DRIVE, "--transfers", "4", check=False)
    if r.returncode != 0:
        print("Aviso: falló la subida al Drive (se reintenta en la siguiente):", r.stderr[-500:])


def _sincronizar(cada=600):
    while not _detener.wait(cada):
        subir_al_drive()


threading.Thread(target=_sincronizar, daemon=True).start()
print("Sincronización con el Drive cada 10 minutos: activa.")

## 1. Vectores (GPU; L/14 tarda bastante más que B/32)

In [ ]:
for codificador in CODIFICADORES:
    if codificador not in YA_CALCULADOS:
        !python -m clasificador.codificadores {codificador}
        subir_al_drive()
print("Vectores en el Drive:", rclone("lsf", f"{DRIVE}/vectores", "--include", "/*.npz").stdout)

## 2. Una cabeza por codificador (también serán miembros del ensamble de la Pareja C)

In [ ]:
# entrenar en train, evaluar en VALIDACIÓN y comparar contra base (los 4 débiles salen primero)
for nombre in EXPERIMENTOS:
    !python -m clasificador.entrenar experimentos/{nombre}.yaml
    !python -m clasificador.evaluar {nombre}
    !python -m clasificador.comparar base {nombre}

# modelos/ y predicciones/ al Drive (las usa la Pareja C); resultados/ también, como respaldo
subir_al_drive()
for nombre in EXPERIMENTOS:
    rclone("copy", f"resultados/{nombre}", f"{DRIVE}/resultados/{nombre}", check=False)
_detener.set()
print("Todo subido a", DRIVE_SALIDAS)

## Subir a la rama `A-codificadores`

Se sube a **`A-codificadores`**, nunca a `main`; luego abran el Pull Request (lo revisa alguien de
**otra** pareja). Con los secretos `GITHUB_TOKEN`, `GIT_NOMBRE` y `GIT_CORREO` no pregunta nada (sirve
en *Save & Run All*); sin ellos los pide aquí. Si corrió en segundo plano sin secretos, los resultados
quedan en la salida del notebook (`clasificador/resultados/`) y en el Drive (`resultados/`).

In [ ]:
from getpass import getpass

token = secreto("GITHUB_TOKEN")
nombre_git, correo_git = secreto("GIT_NOMBRE"), secreto("GIT_CORREO")
if EN_SEGUNDO_PLANO and not (token and nombre_git and correo_git):
    print("Sin GITHUB_TOKEN / GIT_NOMBRE / GIT_CORREO en segundo plano: no se sube a GitHub. "
          "Los resultados están en el Drive (resultados/) y en la salida del notebook.")
else:
    nombre_git = nombre_git or input("Tu nombre: ")
    correo_git = correo_git or input("Tu correo de GitHub: ")
    mensaje = "Vectores L/14 y SigLIP y experimentos A7 (desde Kaggle)" if EN_SEGUNDO_PLANO \
        else input("Qué hiciste (mensaje del commit): ")
    token = token or getpass("Token de GitHub: ")
    !git config user.name "{nombre_git}"
    !git config user.email "{correo_git}"
    !git add experimentos resultados clasificador tests
    !git commit -q -m "{RAMA}: {mensaje}"
    !git push -q https://{token}@github.com/{REPO}.git HEAD:{RAMA}
    print(f"Abre el Pull Request: https://github.com/{REPO}/compare/main...{RAMA}")
del token